<a id="top"></a>

# Lab L1303: Write your first eval set

```yaml
title: "Lab L1303: Write your first eval set"
keywords: evaluation, eval case, scorer, runner, outcome, trajectory, regression case, brittleness, Langfuse, dataset, lab
estimated duration: 40
```

> **Lesson:** L13. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L13/objectives.md).
> Problems 1–4 run **offline** (scripted `FakeModel`, no API key). **Problem 5** uploads your set to the cohort's **live Langfuse** — it soft-skips when Langfuse isn't configured, so this notebook still runs end-to-end.

**Learning objectives.** By the end of this lab you can:

- **write an outcome scorer and a trajectory scorer** — score the *answer* and the *path*;
- **turn an L12 failure into a regression case** — a check that fails when the bug is present and passes when it's fixed;
- **recognize and loosen a brittle check** — use the loosest check that still catches the bug;
- **upload your eval set to Langfuse as a Dataset** — the same cases become durable and reusable.

> Solutions: `L1303_lab_solutions.ipynb`. Problems 1–4 need no API key; Problem 5 uses the cohort Langfuse (and soft-skips without it).

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 — an outcome scorer](#2-problem-1--an-outcome-scorer)
- [3. Problem 2 — a trajectory scorer](#3-problem-2--a-trajectory-scorer)
- [4. Problem 3 — a regression case from an L12 failure](#4-problem-3--a-regression-case-from-an-l12-failure)
- [5. Problem 4 — loosen a brittle check](#5-problem-4--loosen-a-brittle-check)
- [6. Problem 5 — upload your eval set to Langfuse](#6-problem-5--upload-your-eval-set-to-langfuse)
- [7. Self-check](#7-self-check)

## 1. Setup

Run this cell once. It imports the frozen `common/` library and a `run_case` that drives the loop with a scripted `FakeModel` — deterministic, no API key. You write **cases** and **scorers**; `run_case` produces the runs.

In [ ]:
from langchain_core.messages import AIMessage

from fluffy_potato_curriculum.common.agent_loop import RunResult, run
from fluffy_potato_curriculum.common.evals import (
    EvalCase,
    EvalResult,
    evaluate,
    tool_calls,
    tool_trajectory,
)
from fluffy_potato_curriculum.common.fake_model import (
    FakeModel,
    text_reply,
    tool_call,
    tool_reply,
)
from fluffy_potato_curriculum.common.tools import TOOLS

# case id -> the model's scripted moves. FakeModel repeats its LAST line (a runaway).
SCRIPTS: dict[str, list[AIMessage]] = {
    "tokyo": [
        tool_reply(tool_call("c1", "calculator", {"expression": "17*23"})),
        tool_reply(tool_call("c2", "lookup", {"city": "Tokyo"})),
        text_reply("17 * 23 is 391, and Tokyo has a population of 37000000."),
    ],
    "tokyo_reworded": [  # correct, but written with commas
        tool_reply(tool_call("c1", "lookup", {"city": "Tokyo"})),
        text_reply("About 37,000,000 people live in Tokyo."),
    ],
    "atlantis_runaway": [  # one line -> repeats -> max_steps
        tool_reply(tool_call("c1", "lookup", {"city": "Atlantis"})),
    ],
    "atlantis_fixed": [  # gives up cleanly -> natural termination
        text_reply("I do not have a population on file for Atlantis."),
    ],
}


def run_case(case: EvalCase) -> RunResult:
    """Run the loop for one case with its scripted FakeModel (fresh each call)."""
    model = FakeModel(SCRIPTS[case.id])
    return run(model, TOOLS, case.inputs["task"], max_steps=case.inputs.get("max_steps", 8))


print("scripts ready:", ", ".join(SCRIPTS))

[↑ Back to top](#top)

## 2. Problem 1 — an outcome scorer

Write `answer_correct`: an **outcome** scorer that reads `run.final_text` and returns an `EvalResult` whose `score` is `True` when the reference answer (`example.reference_outputs["answer"]`) appears in the final text. Use `key="answer_correct"`.

The asserts run it through the harness on the `tokyo` case (which should pass).

In [ ]:
def answer_correct(*, run: RunResult, example: EvalCase) -> EvalResult:
    """Outcome scorer: is the reference answer a substring of the final text?"""
    # TODO: read the expected answer from example.reference_outputs["answer"],
    #       and return EvalResult(key="answer_correct", score=<expected in run.final_text>).
    raise NotImplementedError("your code here")


tokyo_case = EvalCase(
    id="tokyo",
    inputs={"task": "What is 17 * 23, and the population of Tokyo?"},
    reference_outputs={"answer": "37000000"},
)
report = evaluate(run_case, [tokyo_case], [answer_correct])
print(report.table())
assert report.pass_rate("tokyo", "answer_correct") == 1.0

[↑ Back to top](#top)

## 3. Problem 2 — a trajectory scorer

For an agent, the *path* matters too. Write `expected_tools`: a **trajectory** scorer that uses `tool_trajectory(run)` (the ordered tool-call names) and returns `score=True` when the path equals `example.reference_outputs["expected_tools"]`. Use `key="expected_tools"`.

The `tokyo` case should call `calculator` then `lookup`.

In [ ]:
def expected_tools(*, run: RunResult, example: EvalCase) -> EvalResult:
    """Trajectory scorer: did the ordered tool calls match the reference path?"""
    # TODO: read example.reference_outputs["expected_tools"], compare it to
    #       tool_trajectory(run), and return EvalResult(key="expected_tools", score=...).
    raise NotImplementedError("your code here")


tokyo_traj_case = EvalCase(
    id="tokyo",
    inputs={"task": "What is 17 * 23, and the population of Tokyo?"},
    reference_outputs={"expected_tools": ["calculator", "lookup"]},
)
report = evaluate(run_case, [tokyo_traj_case], [expected_tools])
print(report.table())
assert report.pass_rate("tokyo", "expected_tools") == 1.0

[↑ Back to top](#top)

## 4. Problem 3 — a regression case from an L12 failure

The headline rule: **trace a failure → write a case that catches it → keep it forever.** In L12 you saw a **runaway**: the same `(tool, args)` repeating, ending in `max_steps`. Turn it into a scorer + a case.

Write `no_runaway`: a trajectory scorer that fails when any `(name, args)` pair repeats in `tool_calls(run)` **or** the run terminated `"max_steps"`. A good regression check is **red when broken, green when fixed** — so it must fail on `atlantis_runaway` and pass on `atlantis_fixed`.

In [ ]:
def no_runaway(*, run: RunResult, example: EvalCase) -> EvalResult:
    """Trajectory scorer: no repeated (tool, args) call, and didn't hit the step cap."""
    # TODO: build a list of (name, tuple(sorted(args.items()))) from tool_calls(run);
    #       set repeated = (it has duplicates); set hit_cap = (run.termination == "max_steps");
    #       return EvalResult(key="no_runaway", score=not (repeated or hit_cap)).
    raise NotImplementedError("your code here")


runaway_case = EvalCase(
    id="atlantis_runaway", inputs={"task": "Population of Atlantis?", "max_steps": 4}
)
fixed_case = EvalCase(id="atlantis_fixed", inputs={"task": "Population of Atlantis?"})
report = evaluate(run_case, [runaway_case, fixed_case], [no_runaway])
print(report.table())
# Red when broken, green when fixed:
assert report.pass_rate("atlantis_runaway", "no_runaway") == 0.0
assert report.pass_rate("atlantis_fixed", "no_runaway") == 1.0

[↑ Back to top](#top)

## 5. Problem 4 — loosen a brittle check

The `tokyo_reworded` run is **correct** but writes `37,000,000` with commas — so your `answer_correct` (a plain substring) fails it. A red on a correct run is worse than no check: it trains you to ignore reds.

Write `answer_correct_loose`: same idea, but normalize **commas and spaces** out of both strings before comparing. It should turn `tokyo_reworded` green. Keep `key="answer_correct"`.

In [ ]:
reworded_case = EvalCase(
    id="tokyo_reworded",
    inputs={"task": "What is the population of Tokyo?"},
    reference_outputs={"answer": "37000000"},
)
print("strict check (brittle):")
print(evaluate(run_case, [reworded_case], [answer_correct]).table())


def answer_correct_loose(*, run: RunResult, example: EvalCase) -> EvalResult:
    """Outcome scorer, loosened: compare with commas and spaces normalized out."""
    # TODO: write a small normalizer that removes "," and " " from a string, apply it
    #       to both the expected answer and run.final_text, then substring-compare.
    raise NotImplementedError("your code here")


print("\nloosened check:")
report = evaluate(run_case, [reworded_case], [answer_correct_loose])
print(report.table())
assert report.pass_rate("tokyo_reworded", "answer_correct") == 1.0

**TODO (written):** Name one quality of an answer that *no* string-normalizing check could score — something you'd need an LLM-as-judge (or a human) for. Edit this cell with your answer.

_TODO_

[↑ Back to top](#top)

## 6. Problem 5 — upload your eval set to Langfuse

You built cases and scorers by hand. Now make the *set* durable: collect your cases into one list and upload it as a Langfuse **Dataset** with `upload_dataset(client, cases, name=...)`. That is the same list, hosted — every later run (a different model in `L1305`, the LangGraph agent in L11) scores against it.

Assemble `my_cases` (unique ids!), build a Langfuse client from `require_langfuse()`, and call `upload_dataset`. Gate on `langfuse_configured()` so the cell still runs without keys.

In [ ]:
from fluffy_potato_curriculum.common.config import langfuse_configured, require_langfuse
from fluffy_potato_curriculum.common.evals import upload_dataset

# TODO: collect the cases you built above into ONE list, each with a UNIQUE id
#       (the id is the dataset key). Include your outcome case, your regression
#       case, and your reworded case.
my_cases: list[EvalCase] = []  # TODO: fill this in

# Gate on langfuse_configured() so this cell still runs without keys.
if langfuse_configured():
    from langfuse import Langfuse

    host, public_key, secret_key = require_langfuse()
    client = Langfuse(host=host, public_key=public_key, secret_key=secret_key)
    # TODO: upload_dataset(client, my_cases, name="l13-lab-my-evals"), then client.flush()
    raise NotImplementedError("your code here")
else:
    print("Langfuse not configured — skipping upload. Your eval set:")
    for c in my_cases:
        print(" -", c.id, "->", c.reference_outputs)

[↑ Back to top](#top)

## 7. Self-check

Compare against `L1303_lab_solutions.ipynb`. Done when:

- **Problem 1** `answer_correct` passes the `tokyo` case (`1/1`).
- **Problem 2** `expected_tools` passes the `tokyo` trajectory case (`1/1`).
- **Problem 3** `no_runaway` is **red on `atlantis_runaway`** (`0/1`) and **green on `atlantis_fixed`** (`1/1`) — a real regression case.
- **Problem 4** the strict check fails `tokyo_reworded`, your `answer_correct_loose` turns it green, and your written answer names a quality only a judge/human can score (e.g. tone, "did it acknowledge the failure gracefully", factual correctness beyond the reference).
- **Problem 5** `my_cases` has unique ids and `upload_dataset` runs — creating the Langfuse dataset when configured, or printing your set when not.

You can now write outcome and trajectory scorers, grow a regression case from a real failure, and store an eval set as a Langfuse dataset. In **L1305** you'll run an eval set many times and read **pass rates**.

[↑ Back to top](#top)